# 03 — Hypothesis Testing and Statistical Validation

**Project:** Sigmoid Bike-Sharing Case Study

**Objective:** Statistically validate selected patterns from Notebook 02.

We use daily observations as the unit of analysis where possible, avoiding the mistake of treating 24 hourly records from the same day as independent observations.

**Limitation:** The dataset records completed rentals. It does not contain failed rental attempts or station-level availability.

## 1. Import libraries

In [ ]:
from pathlib import Path
from itertools import combinations

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")
print("Libraries imported successfully.")

## 2. Load datasets

In [ ]:
def find_project_folder():
    locations = [Path.cwd(), *Path.cwd().parents]
    for location in locations:
        for folder in [location, location / "Sigmoid_Bike_Sharing_Case_Study"]:
            if ((folder / "data" / "hour.csv").is_file()
                and (folder / "data" / "day.csv").is_file()):
                return folder

    for folder in [
        Path.home() / "Documents" / "Sigmoid_Bike_Sharing_Case_Study",
        Path.home() / "Downloads" / "Sigmoid_Bike_Sharing_Case_Study",
        Path.home() / "Desktop" / "Sigmoid_Bike_Sharing_Case_Study"
    ]:
        if ((folder / "data" / "hour.csv").is_file()
            and (folder / "data" / "day.csv").is_file()):
            return folder

    import tkinter as tk
    from tkinter import filedialog

    root = tk.Tk()
    root.withdraw()
    root.attributes("-topmost", True)
    selected = filedialog.askdirectory(
        title="Select the Sigmoid_Bike_Sharing_Case_Study folder"
    )
    root.destroy()

    folder = Path(selected) if selected else None
    if folder and (folder / "data" / "hour.csv").is_file() and (folder / "data" / "day.csv").is_file():
        return folder

    raise FileNotFoundError("Could not locate data/hour.csv and data/day.csv.")

In [ ]:
BASE_DIR = find_project_folder()
DATA_DIR = BASE_DIR / "data"

hour_df = pd.read_csv(DATA_DIR / "hour.csv", parse_dates=["dteday"])
day_df = pd.read_csv(DATA_DIR / "day.csv", parse_dates=["dteday"])

hour_work = hour_df.copy()
day_work = day_df.copy()

hour_work["day_type"] = np.where(
    hour_work["workingday"] == 1, "Working day", "Non-working day"
)

print("Project folder:", BASE_DIR)
print("Hourly dataset:", hour_work.shape)
print("Daily dataset:", day_work.shape)

## 3. Prepare complete-day data and analysis windows

In [ ]:
hours_per_day = hour_work.groupby("dteday")["hr"].nunique()
complete_dates = hours_per_day[hours_per_day == 24].index

complete_hour_work = hour_work[
    hour_work["dteday"].isin(complete_dates)
].copy()

AM_HOURS = [7, 8, 9]
PM_HOURS = [17, 18, 19]
MIDDAY_HOURS = [11, 12, 13, 14, 15]
COMMUTE_HOURS = AM_HOURS + PM_HOURS

print("Complete days:", len(complete_dates))
print("Working days:", complete_hour_work.groupby("dteday")["workingday"].first().eq(1).sum())
print("Non-working days:", complete_hour_work.groupby("dteday")["workingday"].first().eq(0).sum())

## 4. Statistical testing functions

Mann–Whitney U compares two independent groups without assuming normality. Cliff's delta measures the direction and magnitude of pairwise differences. Positive values mean group A tends to have larger values than group B.

In [ ]:
def cliffs_delta(group_a, group_b):
    a = np.asarray(group_a)
    b = np.asarray(group_b)
    differences = a[:, None] - b[None, :]
    return (
        np.sum(differences > 0) - np.sum(differences < 0)
    ) / (len(a) * len(b))


def mann_whitney_summary(group_a, group_b, alternative="two-sided"):
    a = pd.Series(group_a).dropna()
    b = pd.Series(group_b).dropna()

    test = stats.mannwhitneyu(
        a, b, alternative=alternative, method="auto"
    )

    return {
        "n_group_a": len(a),
        "n_group_b": len(b),
        "mean_group_a": a.mean(),
        "mean_group_b": b.mean(),
        "median_group_a": a.median(),
        "median_group_b": b.median(),
        "U_statistic": test.statistic,
        "p_value": test.pvalue,
        "cliffs_delta": cliffs_delta(a, b)
    }


def holm_adjust(p_values):
    p_values = np.asarray(p_values, dtype=float)
    n = len(p_values)
    order = np.argsort(p_values)
    adjusted = np.empty(n)
    running_max = 0.0

    for rank, index in enumerate(order):
        running_max = max(running_max, (n - rank) * p_values[index])
        adjusted[index] = min(running_max, 1.0)

    return adjusted

## 5. Hypothesis 1 — Is commute-window share higher on working days?

- **H₀:** Commute-window share is the same on working and non-working days.
- **H₁:** Commute-window share is higher on working days.

Unit of analysis: one complete day. Commute windows: 07:00–09:00 and 17:00–19:00.

In [ ]:
daily_totals = (
    complete_hour_work
    .groupby(["dteday", "workingday"])["cnt"]
    .sum()
    .rename("daily_rentals")
)

commute_totals = (
    complete_hour_work[
        complete_hour_work["hr"].isin(COMMUTE_HOURS)
    ]
    .groupby(["dteday", "workingday"])["cnt"]
    .sum()
    .rename("commute_rentals")
)

daily_commute = pd.concat(
    [daily_totals, commute_totals], axis=1
).reset_index()

daily_commute["commute_rentals"] = (
    daily_commute["commute_rentals"].fillna(0)
)

daily_commute["day_type"] = np.where(
    daily_commute["workingday"] == 1,
    "Working day", "Non-working day"
)

daily_commute["commute_share_pct"] = (
    daily_commute["commute_rentals"]
    / daily_commute["daily_rentals"] * 100
)

display(daily_commute.head())

In [ ]:
working_share = daily_commute.loc[
    daily_commute["workingday"] == 1, "commute_share_pct"
]
nonworking_share = daily_commute.loc[
    daily_commute["workingday"] == 0, "commute_share_pct"
]

h1_result = mann_whitney_summary(
    working_share, nonworking_share, alternative="greater"
)

display(pd.Series(h1_result, name="H1 result").to_frame())

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=daily_commute, x="day_type", y="commute_share_pct")
plt.title("Commute-Window Share by Day Type")
plt.xlabel("Day Type")
plt.ylabel("Commute Rentals (% of Daily Rentals)")
plt.tight_layout()
plt.show()

**Interpretation:** If p-value < 0.05, reject H₀ in favour of higher commute-window share on working days. Report sample sizes, group means, p-value and effect size. This tests a difference in observed rental patterns, not a shortage of bikes.

## 6. Hypothesis 2 — Does demand differ by day type within AM, PM and midday periods?

For each time window, calculate one total per day and compare working with non-working days using a two-sided Mann–Whitney U test.

In [ ]:
def daily_window_totals(data, hours, window_name):
    selected = data[data["hr"].isin(hours)]
    result = (
        selected.groupby(["dteday", "workingday"])["cnt"]
        .sum()
        .reset_index(name="rentals")
    )
    result["window"] = window_name
    return result


am_daily = daily_window_totals(complete_hour_work, AM_HOURS, "AM peak")
pm_daily = daily_window_totals(complete_hour_work, PM_HOURS, "PM peak")
midday_daily = daily_window_totals(complete_hour_work, MIDDAY_HOURS, "Midday")

window_data = pd.concat(
    [am_daily, pm_daily, midday_daily], ignore_index=True
)

window_data["day_type"] = np.where(
    window_data["workingday"] == 1, "Working day", "Non-working day"
)

display(window_data.head())

In [ ]:
h2_results = []

for window in ["AM peak", "PM peak", "Midday"]:
    subset = window_data[window_data["window"] == window]

    working_values = subset.loc[subset["workingday"] == 1, "rentals"]
    nonworking_values = subset.loc[subset["workingday"] == 0, "rentals"]

    result = mann_whitney_summary(
        working_values, nonworking_values, alternative="two-sided"
    )
    result["window"] = window
    h2_results.append(result)

h2_results_df = pd.DataFrame(h2_results)

display(
    h2_results_df[
        ["window", "n_group_a", "n_group_b",
         "mean_group_a", "mean_group_b",
         "U_statistic", "p_value", "cliffs_delta"]
    ]
)

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(
    data=window_data,
    x="window",
    y="rentals",
    hue="day_type",
    order=["AM peak", "PM peak", "Midday"]
)
plt.title("Daily Rentals by Time Window and Day Type")
plt.xlabel("Time Window")
plt.ylabel("Rentals in Window per Day")
plt.tight_layout()
plt.show()

## 7. Multiple-testing correction for Hypothesis 2

Three time windows mean three related tests. Apply Holm correction to reduce the chance of false positives.

In [ ]:
h2_results_df["p_value_holm"] = holm_adjust(
    h2_results_df["p_value"]
)

h2_results_df["significant_after_holm"] = (
    h2_results_df["p_value_holm"] < 0.05
)

display(
    h2_results_df[
        ["window", "p_value", "p_value_holm",
         "significant_after_holm", "cliffs_delta"]
    ]
)

## 8. Hypothesis 3 — Is observed daily demand higher in clear weather?

- **H₀:** Daily demand is not higher in clear weather than in adverse weather.
- **H₁:** Daily demand is higher in clear weather.

Weather category 1 is treated as clear/few clouds; categories 2 and 3 are grouped as adverse weather. Category 4 remains separate because it has very few observations.

In [ ]:
day_weather = day_work.copy()

day_weather["weather_group"] = np.select(
    [
        day_weather["weathersit"] == 1,
        day_weather["weathersit"].isin([2, 3]),
        day_weather["weathersit"] == 4
    ],
    [
        "Clear",
        "Adverse (categories 2-3)",
        "Heavy (category 4)"
    ],
    default="Unknown"
)

display(
    day_weather["weather_group"]
    .value_counts()
    .to_frame("number_of_days")
)

In [ ]:
clear_demand = day_weather.loc[
    day_weather["weather_group"] == "Clear", "cnt"
]
adverse_demand = day_weather.loc[
    day_weather["weather_group"] == "Adverse (categories 2-3)", "cnt"
]

h3_result = mann_whitney_summary(
    clear_demand, adverse_demand, alternative="greater"
)

display(pd.Series(h3_result, name="H3 result").to_frame())

In [ ]:
weather_summary = (
    day_weather.groupby("weather_group")["cnt"]
    .agg(["count", "mean", "median", "std"])
)

display(weather_summary)

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(
    data=day_weather,
    x="weather_group",
    y="cnt",
    order=["Clear", "Adverse (categories 2-3)", "Heavy (category 4)"]
)
plt.title("Daily Rentals by Weather Group")
plt.xlabel("Weather Group")
plt.ylabel("Daily Rentals")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

**Interpretation:** A significant result indicates an association between weather group and completed daily rentals. Category 4 has a very small sample and should not support a strong standalone conclusion.

## 9. Hypothesis 4 — Does daily demand differ across season codes?

- **H₀:** Daily demand distributions are the same across the four season codes.
- **H₁:** At least one season-code group differs.

Use the Kruskal–Wallis test. Keep numeric season codes until the dataset README mapping has been verified.

In [ ]:
season_codes = sorted(day_work["season"].dropna().unique())

season_groups = [
    day_work.loc[day_work["season"] == season, "cnt"].dropna()
    for season in season_codes
]

season_test = stats.kruskal(*season_groups)

season_result = {
    "test": "Kruskal-Wallis",
    "number_of_groups": len(season_groups),
    "H_statistic": season_test.statistic,
    "p_value": season_test.pvalue
}

display(pd.Series(season_result, name="Season test").to_frame())

In [ ]:
season_summary = (
    day_work.groupby("season")["cnt"]
    .agg(["count", "mean", "median", "std"])
)

display(season_summary)

In [ ]:
plt.figure(figsize=(9, 5))
sns.boxplot(data=day_work, x="season", y="cnt")
plt.title("Daily Rentals by Season Code")
plt.xlabel("Season Code")
plt.ylabel("Daily Rentals")
plt.tight_layout()
plt.show()

## 10. Pairwise seasonal comparisons with Holm correction

Only run pairwise comparisons if the overall Kruskal–Wallis test is significant. Apply Holm correction across the pairwise tests.

In [ ]:
pairwise_season_results = []

if season_test.pvalue < 0.05:
    for season_a, season_b in combinations(season_codes, 2):
        values_a = day_work.loc[day_work["season"] == season_a, "cnt"]
        values_b = day_work.loc[day_work["season"] == season_b, "cnt"]

        test = stats.mannwhitneyu(
            values_a, values_b, alternative="two-sided"
        )

        pairwise_season_results.append({
            "season_a": season_a,
            "season_b": season_b,
            "mean_a": values_a.mean(),
            "mean_b": values_b.mean(),
            "U_statistic": test.statistic,
            "p_value": test.pvalue
        })
else:
    print("Overall seasonal test is not significant; pairwise tests skipped.")

pairwise_season_df = pd.DataFrame(pairwise_season_results)

if not pairwise_season_df.empty:
    pairwise_season_df["p_value_holm"] = holm_adjust(
        pairwise_season_df["p_value"]
    )
    pairwise_season_df["significant_after_holm"] = (
        pairwise_season_df["p_value_holm"] < 0.05
    )

display(pairwise_season_df)

## 11. Hypothesis test summary

The summary below includes H1, all three H2 time-window comparisons, H3 and H4.

For H2, decisions use the **Holm-adjusted p-values** because three related time-window tests are performed. H1, H3 and H4 are shown with their raw p-values; they are not adjusted together with H2.

In [ ]:
summary_rows = [
    {
        "hypothesis": "H1: Commute share higher on working days",
        "test": "One-sided Mann-Whitney U",
        "p_value": h1_result["p_value"],
        "adjusted_p_value": np.nan,
        "effect_size": h1_result["cliffs_delta"],
        "decision": "Reject H0" if h1_result["p_value"] < 0.05 else "Fail to reject H0"
    }
]

for _, row in h2_results_df.iterrows():
    summary_rows.append({
        "hypothesis": f"H2: {row['window']} demand differs by day type",
        "test": "Two-sided Mann-Whitney U",
        "p_value": row["p_value"],
        "adjusted_p_value": row["p_value_holm"],
        "effect_size": row["cliffs_delta"],
        "decision": (
            "Reject H0"
            if row["p_value_holm"] < 0.05
            else "Fail to reject H0"
        )
    })

summary_rows.extend([
    {
        "hypothesis": "H3: Clear-weather demand higher than adverse",
        "test": "One-sided Mann-Whitney U",
        "p_value": h3_result["p_value"],
        "adjusted_p_value": np.nan,
        "effect_size": h3_result["cliffs_delta"],
        "decision": "Reject H0" if h3_result["p_value"] < 0.05 else "Fail to reject H0"
    },
    {
        "hypothesis": "H4: Demand differs across season codes",
        "test": "Kruskal-Wallis",
        "p_value": season_test.pvalue,
        "adjusted_p_value": np.nan,
        "effect_size": np.nan,
        "decision": "Reject H0" if season_test.pvalue < 0.05 else "Fail to reject H0"
    }
])

hypothesis_summary = pd.DataFrame(summary_rows)

display(hypothesis_summary)

## 12. What the tests can and cannot establish

### What they can establish
- Whether commute-window share differs between working and non-working days.
- Whether rentals in selected time windows differ by day type.
- Whether observed daily rentals are associated with weather category.
- Whether demand distributions differ across season codes.

### What they cannot establish
- Whether a rider arrived and failed to find a bike.
- Which station experienced a stock-out.
- Whether the total fleet was insufficient.
- Whether redistribution would causally improve availability.

These questions require station-level inventory snapshots, failed-rental records, trip origin/destination data, and an operational experiment.

### Reporting checklist
For every test, report the hypotheses, unit of analysis, sample sizes, test statistic, p-value, effect size, correction for multiple tests, and practical interpretation.

### Next notebook
**`04_demand_forecasting.ipynb`** — chronological train/test split, baseline models, machine-learning models, evaluation metrics, and error analysis.